### This Notebook demonstrates how to generate code from RAG docs.
we are using langchain python and js documentation as RAG.
We will use the following steps:
1. Load the documents
2. Create a vector store
3. Generate code from the documents
4. Save the generated code
5. Run the generated code
6. Display the output of the generated code


Step 0: setup (* installation in  requirements.txt and env examples for set env variables)

In [10]:
import os
from dotenv import load_dotenv

load_dotenv()
GoogleGeminiKey = os.getenv('GOOGLE_GEMINI_KEY')
connection = os.getenv("PG_CONNECTION_STRING")

Step 1: Create database

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
from langchain_community.document_loaders import WebBaseLoader
import bs4


def clean_text(text):
    return re.sub(r'[\u200b\u200e\u200f]', '', text).strip()


def extract_how_tos(link: str):
    response = requests.get(link)
    soup = BeautifulSoup(response.text, "html.parser")
    content_div = soup.find("div", class_="theme-doc-markdown")

    results = []
    current_module = None

    for tag in content_div.find_all(["h2", "h3", "ul"]):
        if tag.name in ["h2", "h3"]:
            current_module = {
                "module": clean_text(tag.get_text()),
                "how_tos": []
            }
            results.append(current_module)

        elif tag.name == "ul" and current_module:
            for li in tag.find_all("li"):
                a = li.find("a")
                if a and a.get("href"):
                    link = urljoin(BASE_URL, a["href"])
                    current_module["how_tos"].append({
                        "name": clean_text(a.get_text()),
                        "link": link
                    })

    return results


async def web_base_loader(page_urls):
    loader = WebBaseLoader(
        web_paths=page_urls,
        bs_kwargs={
            "parse_only": bs4.SoupStrainer(class_="theme-doc-markdown markdown"),
        },
        # bs_get_text_kwargs={"separator": " | ", "strip": True}
    )
    documents = []

    async for doc in loader.alazy_load():
        documents.append(doc)

    # assert len(documents) == 1, "No documents loaded from the web page"
    # print("Web page loaded successfully!")
    return documents

In [14]:
# for python docs:

BASE_URL = "https://python.langchain.com"
HOW_TO_URL = f"{BASE_URL}/docs/how_to/"

python_langchain_tutorial_links = extract_how_tos(HOW_TO_URL) 


In [66]:
# Summary chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI
from langchain_core.runnables import chain
from langchain_core.pydantic_v1 import BaseModel, Field


class PageDetails(BaseModel):
    "Document Web page summarization and meta data extraction."
    summary: str
    description: str
    tags: list[str]


summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are professional in summarizing documents.And having 20+ years of experience in Programming and development. So you can easily and efficiently summarize the given documentation of any module. You are also good at understanding and summarizing complex technical documents."),
    ("system", "Summary should not be long, and Also generate the short description of 50 words about the documentation and list of tags."),
    ("user",
     "Summarize the following documentation of Langchain for module {module}, document: {document}"),
])

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.1,  # Allows 1 request every 10 seconds
    check_every_n_seconds=0.1,  # Checks every 100 ms
    max_bucket_size=10  # Maximum burst size
)

llm_lite = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite", api_key=GoogleGeminiKey,
                                  temperature=0, rate_limiter=rate_limiter).with_structured_output(PageDetails)


def summary_generator(module, document):
    summary_chain = summary_prompt | llm_lite
    summary = summary_chain.invoke({"module": module, "document": document})
    return summary

In [ ]:
dc = python_langchain_tutorial_links[0]
doc = await web_base_loader([d['link'] for d in dc['how_tos']])

Fetching pages: 100%|##########| 2/2 [00:00<00:00,  9.84it/s]


In [58]:
summary_generator(dc['module'], doc[0])

PageDetails(summary='The LangChain ecosystem offers different packages for specific functionalities. The main package is installed with pip install langchain. Other packages include langchain-core, integration packages (e.g., langchain-openai), langchain-community, langchain-experimental, LangGraph, LangServe, LangChain CLI, and LangSmith SDK. Installation instructions are provided for each, including from source.', description='This document provides instructions on how to install various LangChain packages. It covers the main langchain package, ecosystem packages, integration packages, experimental packages, LangGraph, LangServe, LangChain CLI, and LangSmith SDK. It also explains how to install packages from source.', tags=['LangChain', 'installation', 'packages', 'pip', 'conda'])

In [ ]:
documents = []
summaries = []
for how_tos in python_langchain_tutorial_links:
    module = how_tos['module']
    print(f"Module: {module} , links : {len(how_tos['how_tos'])}")
    # for how_to in module['how_tos']:
    #     print(f"  - How To: {how_to['name']}")
    #     print(f"    Link: {how_to['link']}")
    links_in_module = [d['link'] for d in how_tos['how_tos']]
    titles = [d['name'] for d in how_tos['how_tos']]
    
    if(len(links_in_module) > 0):
        docs = await web_base_loader(links_in_module)
        for index,doc in enumerate(docs):
            output = summary_generator(module, doc)
            summaries.append(output.summary)
            doc.metadata['description'] = output.description
            doc.metadata['tags'] = output.tags
            doc.metadata['title'] = titles[index]
            doc.metadata['module'] = module
            documents.append(doc)
    else:
        print(f"No links found for module: {module}")

Module: Components , links : 0
No links found for module: Components
Module: Chat models , links : 17


Fetching pages: 100%|##########| 17/17 [00:02<00:00,  7.36it/s]


Module: Messages , links : 3


Fetching pages: 100%|##########| 3/3 [00:00<00:00,  4.84it/s]


Module: Prompt templates , links : 5


...

Fetching pages: 100%|##########| 3/3 [00:00<00:00,  5.12it/s]


Module: LangChain Expression Language (LCEL) , links : 13


Fetching pages: 100%|##########| 13/13 [00:01<00:00,  8.09it/s]


Module: LangGraph , links : 0
No links found for module: LangGraph
Module: LangSmith , links : 0
No links found for module: LangSmith
Module: Evaluation , links : 0
No links found for module: Evaluation
Module: Tracing , links : 2


Fetching pages: 100%|##########| 2/2 [00:00<00:00,  5.84it/s]


In [75]:
len(summaries),len(documents),documents[0].metadata 

(161,
 161,
 {'source': 'https://python.langchain.com/docs/how_to/installation/',
  'description': 'This document provides instructions on how to install various LangChain packages, including the main langchain package, integration packages, experimental packages, and other related tools like LangGraph, LangServe, LangChain CLI, and LangSmith SDK. It also covers installation from source.',
  'tags': ['LangChain', 'Installation', 'pip', 'conda', 'packages'],
  'title': 'How to: install LangChain packages',
  'module': 'Installation'})

### indexing and save to local and PGVector DB

In [78]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001", google_api_key=GoogleGeminiKey)

In [83]:
from langchain.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
import faiss

index = faiss.IndexFlatL2(len(embeddings.embed_query("hello world")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
from uuid import uuid4
from langchain_core.documents import Document
id_key="doc_id"
doc_ids = [str(uuid4()) for _ in range(len(documents))]
summary_doc = [
    Document(page_content=s,metadata={id_key:doc_ids[i]}) for i,s in enumerate(summaries)
]
vector_store.add_documents(summary_doc)
vector_store.save_local("summaries_store_index")

['9bab3f49-056d-4b05-aa64-9f09c17540f0',
 'ad216864-1a81-480c-8d06-c5eac3ac6078',
 'fd7ce2a4-a89f-4274-a8b7-19378a61aa47',
 '18fc6be1-ff8f-4faf-a3b0-dc90d9f83ab9',
 'a079c708-e46b-4fc8-b975-0353212e6054',
 '82d73d47-dab5-40ee-b8f6-cd06e61e454b',
 'dbfa998b-e2f7-413e-8741-5fca7ff3cad4',
 '2489115d-869a-44df-a02f-99461344b0e1',
 '83dfcf37-03b3-4101-84c1-8e67d093c9aa',
 '30267cca-d18a-4971-b9e5-f147690682a4',
 '7c00471c-a4b1-463f-996e-13f28884aa06',
 'd5cfb03c-2e13-4806-82ef-acbcce51efc7',
 '58de586b-0ee1-46e3-8570-1147adfb1a23',
 'bf7ba9d1-a6a3-4c0b-a43d-7793604f8e55',
 'ae04a403-3035-4654-a7c4-7c565bbb71c7',
 '9ddca67a-51d1-4df9-8871-d4d24be674f2',
 'bd858ede-a417-42a1-a592-303c203a1803',
 '62d6b629-e6e6-4032-8555-f52415da43e1',
 'cc4b844a-b8bd-490b-b56e-33033bc339ce',
 '0972526f-643b-4a77-a0de-9095ccd483d7',
 '6f20e7f6-3c47-440b-a883-f3d352392977',
 '71e823ae-9557-4cbb-b9c6-e8ea36bd4cd0',
 '7d2a9309-9639-4ba1-80ab-4a0209c49209',
 'daa206c4-d90e-4f9b-b108-ed921dfce549',
 'ba5d254c-b019-

In [93]:
from langchain.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
import faiss

index2 = faiss.IndexFlatL2(len(embeddings.embed_query("document_index")))

vector_store_document = FAISS(
    embedding_function=embeddings,
    index=index2,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
vector_store_document.add_documents(documents=documents,ids=doc_ids)
vector_store_document.save_local("documents_store_index")



In [94]:
python_how_to_doc_collection_name = "LangChain_Python_how_tos"
js_how_to_doc_collection_name = "LangChain_JavaScript_how_tos"


In [97]:
## PGVector 
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore
from langchain_postgres.vectorstores import PGVector

# the vector store to use to index the child chunks
vector_store_py = PGVector(
    embeddings=embeddings,
    collection_name=python_how_to_doc_collection_name,
    connection=connection,
    use_jsonb=True,
)

store_py = InMemoryStore()

retriever_py = MultiVectorRetriever( 
    vectorstore=vector_store_py,
    docstore=store_py,
    id_key=id_key,
)


# add the document summaries to the vector store for similarity search:
retriever_py.vectorstore.add_documents(summary_doc)

# Store the original document in the store, linked to the summaries via doc_id
# this allows us to first search summaries efficiently, then fetch the full docs when needed
retriever_py.docstore.mset(list(zip(doc_ids,documents)))

#### same for JS

In [102]:
BASE_URL = "https://js.langchain.com"
HOW_TO_URL = f"{BASE_URL}/docs/how_to/"

js_langchain_tutorial_links = extract_how_tos(HOW_TO_URL) 

documents_js = []
summaries_js = []
for how_tos in js_langchain_tutorial_links:
    module = how_tos['module']
    print(f"Module: {module} , links : {len(how_tos['how_tos'])}")
    # for how_to in module['how_tos']:
    #     print(f"  - How To: {how_to['name']}")
    #     print(f"    Link: {how_to['link']}")
    links_in_module = [d['link'] for d in how_tos['how_tos']]
    titles = [d['name'] for d in how_tos['how_tos']]
    
    if(len(links_in_module) > 0):
        docs = await web_base_loader(links_in_module)
        for index,doc in enumerate(docs):
            output = summary_generator(module, doc)
            summaries_js.append(output.summary)
            doc.metadata['description'] = output.description
            doc.metadata['tags'] = output.tags
            doc.metadata['title'] = titles[index]
            doc.metadata['module'] = module
            documents_js.append(doc)
    else:
        print(f"No links found for module: {module}")

Module: Installation , links : 1


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  4.84it/s]


Module: Key features , links : 4


Fetching pages: 100%|##########| 4/4 [00:00<00:00,  8.52it/s]


Module: LangChain Expression Language (LCEL) , links : 11


...

Fetching pages: 100%|##########| 3/3 [00:00<00:00,  5.55it/s]


Module: Q&A over graph databases , links : 4


Fetching pages: 100%|##########| 4/4 [00:00<00:00,  5.47it/s]


Module: LangGraph.js , links : 0
No links found for module: LangGraph.js
Module: LangSmith , links : 0
No links found for module: LangSmith
Module: Evaluation , links : 0
No links found for module: Evaluation
Module: Tracing , links : 2


Fetching pages: 100%|##########| 2/2 [00:00<00:00,  5.45it/s]


In [105]:
from langchain.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
import faiss
from uuid import uuid4
from langchain_core.documents import Document

index_js = faiss.IndexFlatL2(len(embeddings.embed_query("js_langchain_summary_store")))

vector_store_js = FAISS(
    embedding_function=embeddings,
    index=index_js,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

id_key="doc_id"
doc_ids_js = [str(uuid4()) for _ in range(len(documents_js))]
summary_doc_js = [
    Document(page_content=s,metadata={id_key:doc_ids_js[i]}) for i,s in enumerate(summaries_js)
]
vector_store_js.add_documents(summary_doc_js)
vector_store_js.save_local("summaries_store_index_js")

index_docs_js = faiss.IndexFlatL2(len(embeddings.embed_query("js_langchain_document_store")))

vector_store_doc_js = FAISS(
    embedding_function=embeddings,
    index=index_docs_js,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

vector_store_doc_js.add_documents(documents=documents_js,ids=doc_ids_js)
vector_store_doc_js.save_local("document_store_index_js")

In [107]:
## PGVector 
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore
from langchain_postgres.vectorstores import PGVector

# the vector store to use to index the child chunks
vector_store_js = PGVector(
    embeddings=embeddings,
    collection_name=js_how_to_doc_collection_name,
    connection=connection,
    use_jsonb=True,
)

store_js = InMemoryStore()

retriever_js = MultiVectorRetriever( 
    vectorstore=vector_store_js,
    docstore=store_js,
    id_key=id_key,
)


# add the document summaries to the vector store for similarity search:
retriever_js.vectorstore.add_documents(summary_doc_js)

# Store the original document in the store, linked to the summaries via doc_id
# this allows us to first search summaries efficiently, then fetch the full docs when needed
retriever_js.docstore.mset(list(zip(doc_ids_js,documents_js)))

Exception: Failed to create vector extension: (psycopg.errors.ConnectionTimeout) connection timeout expired
Multiple connection attempts failed. All failures were:
- host: 'localhost', port: 6024, hostaddr: '::1': connection failed: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.
- host: 'localhost', port: 6024, hostaddr: '127.0.0.1': connection timeout expired
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [109]:
from typing import Literal
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

class QueryRoute(BaseModel):
    "Route a user query to the most relevant datasource"
    datasource: Literal["python_docs","js_docs"] = Field(...,description="Given a user question, Choose which datasource is most relevant to the question")


llm_with_struct_output = ChatGoogleGenerativeAI(model="gemini-2.0-flash-001",
                         api_key=GoogleGeminiKey, temperature=0).with_structured_output(QueryRoute)

system_prompt = """
you are an expert query router that routes user queries to the most relevant datasource based on the question. 
Based on the programming language the question is referring to, route to the most relevant datasource.

If the question not mention in which programming language, default to python. 
"""

router_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("user", "{query}"),
])

router_chain = router_prompt | llm_with_struct_output


In [112]:
router_chain.invoke("How to create rag chain at with lang chain in js?")

QueryRoute(datasource='js_docs')

In [ ]:
results  = vector_store_document.similarity_search(
    "How to use intall lang chain",
    k=2,
)

for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* How to route execution within a chainPrerequisitesThis guide assumes familiarity with the following concepts:LangChain Expression Language (LCEL)Chaining runnablesConfiguring chain parameters at runtimePrompt templatesChat MessagesThis guide covers how to do routing in the LangChain Expression Language.Routing allows you to create non-deterministic chains where the output of a previous step defines the next step. Routing helps provide structure and consistency around interactions with LLMs.There are two ways to perform routing:Conditionally return runnables from a RunnableLambda (recommended)Using a RunnableBranch (legacy)We'll illustrate both methods using a two step sequence where the first step classifies an input question as being about LangChain, Anthropic, or Other, then routes to a corresponding prompt chain.Using a custom function​You can use a custom function to route between different outputs. Here's an example:tipSee this section for general instructions on installing inte

In [129]:
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate

sys_prompt = """
You are an expert programming assistance and generate accurate code by reading documentation and example code.
and summarizing it by joining all the pieces of code into a single coherent piece of code.
And also provide the explanation of the code.
Generate result in markdown format.
if you there is no document available the say: I do not have any references Buy from by knowledge it can be done in that way and add your solution.
"""

code_prompt = ChatPromptTemplate.from_messages([
    ("system", sys_prompt),
    ("user",
     "Provide the code in {data_source} and explanation. For the user query: {query} from the following documentation: {docs}")
])

# 1. Wrap the router to include both input and output
wrapped_router = RunnableLambda(
    lambda x: {"query": x, "result": router_chain.invoke(x)})

code_llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-001",
                         api_key=GoogleGeminiKey, temperature=0.7)


def next_chain(result):
    k = 3
    query = result['query']
    result = result['result']
    print(query, result)
    docs = None
    if "python_docs" in result.datasource.lower():
        print("user is asking about python docs")
        docs = vector_store_document.similarity_search(
            query,
            k=k,
        )
    else:
        docs = vector_store_doc_js.similarity_search(
            query,
            k=k,
        )
        print("user is asking about js docs")

    code_chain = code_prompt | code_llm
    final_output = code_chain.invoke({"query":query,"data_source":result.datasource,"docs":docs})

    return {"docs":docs, "final_output":final_output}


main_chain = wrapped_router | RunnableLambda(next_chain)

In [ ]:
from IPython.display import Markdown, display

result = main_chain.invoke("How to create rag chain app with lang chain in js?")
display(Markdown(result['final_output'].content))

for i in result['docs']:
    print(i.metadata['source'])


```javascript
/**
 * @module RAGChainApp
 * @description This module provides functions to create a Retrieval Augmented Generation (RAG) chain application using LangChain.js.
 */

import { ChatPromptTemplate } from "@langchain/core/prompts";
import { StringOutputParser } from "@langchain/core/output_parsers";
import { RunnableSequence } from "@langchain/core/runnables";
import { ChatAnthropic } from "@langchain/anthropic";
import { OpenAIEmbeddings } from "@langchain/openai";
import { cosineSimilarity } from "@langchain/core/utils/math";
import { MemorySaver } from "@langchain/langgraph";
import { createReactAgent } from "@langchain/langgraph/prebuilt";
import { tool } from "@langchain/core/tools";
import { z } from "zod";
import { AgentExecutor, createToolCallingAgent } from "langchain/agents";
import {
  BaseMessage,
  SystemMessage,
  HumanMessage,
} from "@langchain/core/messages";
import { ChatMessageHistory } from "@langchain/community/stores/message/in_memory";
import { RunnableWithMessageHistory } from "@langchain/core/runnables";
import { GraphRecursionError } from "@langchain/langgraph";

/**
 * @async
 * @function createSemanticRouterChain
 * @description Creates a chain that routes queries based on semantic similarity to predefined templates.
 * @param {string} modelName - The name of the Anthropic chat model to use.
 * @param {string[]} templateTexts - An array of template texts for different categories (e.g., physics, math).
 * @returns {Promise<RunnableSequence>} A runnable sequence that performs semantic routing and generates responses.
 */
async function createSemanticRouterChain(
  modelName: string,
  templateTexts: string[]
): Promise<RunnableSequence> {
  const embeddings = new OpenAIEmbeddings({});
  const templateEmbeddings = await embeddings.embedDocuments(templateTexts);

  /**
   * @async
   * @function promptRouter
   * @description Routes a query to the most relevant prompt based on cosine similarity.
   * @param {string} query - The input query.
   * @returns {Promise<ChatPromptTemplate>} A promise that resolves to a ChatPromptTemplate.
   */
  const promptRouter = async (query: string): Promise<ChatPromptTemplate> => {
    const queryEmbedding = await embeddings.embedQuery(query);
    const similarity = cosineSimilarity([queryEmbedding], templateEmbeddings)[0];
    const isPhysicsQuestion = similarity[0] > similarity[1];
    let promptTemplate: ChatPromptTemplate;

    if (isPhysicsQuestion) {
      console.log("Using physics prompt");
      promptTemplate = ChatPromptTemplate.fromTemplate(templateTexts[0]);
    } else {
      console.log("Using math prompt");
      promptTemplate = ChatPromptTemplate.fromTemplate(templateTexts[1]);
    }

    return promptTemplate;
  };

  const model = new ChatAnthropic({ model: modelName });
  const chain = RunnableSequence.from([
    promptRouter,
    model,
    new StringOutputParser(),
  ]);

  return chain;
}

/**
 * @async
 * @function createLCELRouterChain
 * @description Creates a chain that routes queries based on a classification step using LangChain Expression Language (LCEL).
 * @param {string} modelName - The name of the Anthropic chat model to use.
 * @returns {Promise<RunnableSequence>} A runnable sequence that performs classification-based routing and generates responses.
 */
async function createLCELRouterChain(modelName: string): Promise<RunnableSequence> {
  const promptTemplate = ChatPromptTemplate.fromTemplate(
    `Given the user question below, classify it as either being about \`LangChain\`, \`Anthropic\`, or \`Other\`. 
        Do not respond with more than one word.
        <question>{question}</question>
        Classification:`
  );

  const model = new ChatAnthropic({ model: modelName });

  const classificationChain = RunnableSequence.from([
    promptTemplate,
    model,
    new StringOutputParser(),
  ]);

  const langChainChain = ChatPromptTemplate.fromTemplate(
    `You are an expert in langchain. Always answer questions starting with "As Harrison Chase told me". Respond to the following question: Question: {question} Answer:`
  ).pipe(model);

  const anthropicChain = ChatPromptTemplate.fromTemplate(
    `You are an expert in anthropic. Always answer questions starting with "As Dario Amodei told me". Respond to the following question: Question: {question} Answer:`
  ).pipe(model);

  const generalChain = ChatPromptTemplate.fromTemplate(
    `Respond to the following question: Question: {question} Answer:`
  ).pipe(model);

  const route = ({ topic }: { input: string; topic: string }) => {
    if (topic.toLowerCase().includes("anthropic")) {
      return anthropicChain;
    }
    if (topic.toLowerCase().includes("langchain")) {
      return langChainChain;
    }
    return generalChain;
  };

  const fullChain = RunnableSequence.from([
    {
      topic: classificationChain,
      question: (input: { question: string }) => input.question,
    },
    route,
  ]);

  return fullChain;
}

/**
 * @async
 * @function createLangGraphAgent
 * @description Creates an agent using LangGraph, demonstrating tool calling and memory.
 * @param {string} modelName - The name of the OpenAI chat model to use.
 * @returns {Promise<object>} An object containing the agent's output messages.
 */
async function createLangGraphAgent(modelName: string): Promise<object> {
  // Define the LLM
  const llm = new ChatOpenAI({
    model: modelName,
  });

  // Define a tool
  const magicTool = tool(
    async ({ input }: { input: number }) => {
      return `${input + 2}`;
    },
    {
      name: "magic_function",
      description: "Applies a magic function to an input.",
      schema: z.object({
        input: z.number(),
      }),
    }
  );
  const tools = [magicTool];

  // Create the agent
  const app = createReactAgent({
    llm,
    tools,
  });

  // Invoke the agent
  const query = "what is the value of magic_function(3)?";
  let agentOutput = await app.invoke({
    messages: [{ role: "user", content: query }],
  });

  // Demonstrate memory
  const checkpointer = new MemorySaver();
  const appWithMemory = createReactAgent({
    llm: llm,
    tools: tools,
    checkpointSaver: checkpointer,
  });

  const langGraphConfig = {
    configurable: {
      thread_id: "test-thread",
    },
  };

  agentOutput = await appWithMemory.invoke(
    {
      messages: [
        {
          role: "user",
          content: "Hi, I'm polly! What's the output of magic_function of 3?",
        },
      ],
    },
    langGraphConfig
  );

  agentOutput = await appWithMemory.invoke(
    {
      messages: [{ role: "user", content: "Remember my name?" }],
    },
    langGraphConfig
  );

  agentOutput = await appWithMemory.invoke(
    {
      messages: [{ role: "user", content: "what was that output again?" }],
    },
    langGraphConfig
  );

  return agentOutput;
}

/**
 * @async
 * @function createToolCallingAgentExample
 * @description Creates a tool-calling agent using LangChain agents.
 * @param {string} modelName - The name of the OpenAI chat model to use.
 * @returns {Promise<object>} An object containing the agent's output.
 */
async function createToolCallingAgentExample(modelName: string): Promise<object> {
  // Define the LLM
  const llm = new ChatOpenAI({ model: modelName });

  // Define a tool
  const magicTool = tool(
    async ({ input }: { input: number }) => {
      return `${input + 2}`;
    },
    {
      name: "magic_function",
      description: "Applies a magic function to an input.",
      schema: z.object({
        input: z.number(),
      }),
    }
  );
  const tools = [magicTool];

  // Define a prompt
  const prompt = ChatPromptTemplate.fromMessages([
    ["system", "You are a helpful assistant"],
    ["placeholder", "{chat_history}"],
    ["human", "{input}"],
    ["placeholder", "{agent_scratchpad}"],
  ]);

  // Create the agent
  const agent = createToolCallingAgent({
    llm,
    tools,
    prompt,
  });

  const agentExecutor = new AgentExecutor({
    agent,
    tools,
  });

  const query = "what is the value of magic_function(3)?";
  const result = await agentExecutor.invoke({ input: query });

  return result;
}

/**
 * @async
 * @function createAgentWithMemory
 * @description Creates an agent with memory using LangChain agents.
 * @param {string} modelName - The name of the OpenAI chat model to use.
 * @returns {Promise<void>}
 */
async function createAgentWithMemory(modelName: string): Promise<void> {
  // Define the LLM
  const llm = new ChatOpenAI({ model: modelName });

  // Define a tool
  const magicTool = tool(
    async ({ input }: { input: number }) => {
      return `${input + 2}`;
    },
    {
      name: "magic_function",
      description: "Applies a magic function to an input.",
      schema: z.object({
        input: z.number(),
      }),
    }
  );
  const tools = [magicTool];

  // Define a prompt
  const prompt = ChatPromptTemplate.fromMessages([
    ["system", "You are a helpful assistant"],
    ["placeholder", "{chat_history}"],
    ["human", "{input}"],
    ["placeholder", "{agent_scratchpad}"],
  ]);

  // Create the agent
  const agent = createToolCallingAgent({
    llm,
    tools,
    prompt,
  });

  const agentExecutor = new AgentExecutor({
    agent,
    tools,
  });

  // Add memory
  const memory = new ChatMessageHistory();
  const agentExecutorWithMemory = new RunnableWithMessageHistory({
    runnable: agentExecutor,
    getMessageHistory: () => memory,
    inputMessagesKey: "input",
    historyMessagesKey: "chat_history",
  });

  const config = { configurable: { sessionId: "test-session" } };
  let agentOutput = await agentExecutorWithMemory.invoke(
    { input: "Hi, I'm polly! What's the output of magic_function of 3?" },
    config
  );
  console.log(agentOutput.output);

  agentOutput = await agentExecutorWithMemory.invoke(
    { input: "Remember my name?" },
    config
  );
  console.log("---");
  console.log(agentOutput.output);
  console.log("---");

  agentOutput = await agentExecutorWithMemory.invoke(
    { input: "what was that output again?" },
    config
  );
  console.log(agentOutput.output);
}

/**
 * @async
 * @function createAgentWithMaxIterations
 * @description Creates an agent with a maximum number of iterations to prevent infinite loops.
 * @param {string} modelName - The name of the OpenAI chat model to use.
 * @returns {Promise<void>}
 */
async function createAgentWithMaxIterations(modelName: string): Promise<void> {
  // Define the LLM
  const llm = new ChatOpenAI({ model: modelName });

  const badMagicTool = tool(
    async ({ input: _input }) => {
      return "Sorry, there was a temporary error. Please try again with the same input.";
    },
    {
      name: "magic_function",
      description: "Applies a magic function to an input.",
      schema: z.object({
        input: z.string(),
      }),
    }
  );
  const badTools = [badMagicTool];

  const spanishPrompt = ChatPromptTemplate.fromMessages([
    ["system", "You are a helpful assistant. Respond only in Spanish."],
    ["placeholder", "{chat_history}"],
    ["human", "{input}"],
    ["placeholder", "{agent_scratchpad}"],
  ]);

  const spanishAgentExecutorWithMaxIterations = new AgentExecutor({
    agent: createToolCallingAgent({
      llm,
      tools: badTools,
      prompt: spanishPrompt,
    }),
    tools: badTools,
    verbose: true,
    maxIterations: 2,
  });

  const query = "what is the value of magic_function(3)?";
  await spanishAgentExecutorWithMaxIterations.invoke({ input: query });
}

/**
 * @async
 * @function createAgentWithRecursionLimit
 * @description Creates an agent with a recursion limit to prevent infinite loops in LangGraph.
 * @param {string} modelName - The name of the OpenAI chat model to use.
 * @returns {Promise<void>}
 */
async function createAgentWithRecursionLimit(modelName: string): Promise<void> {
  // Define the LLM
  const llm = new ChatOpenAI({ model: modelName });

  const badMagicTool = tool(
    async ({ input: _input }) => {
      return "Sorry, there was a temporary error. Please try again with the same input.";
    },
    {
      name: "magic_function",
      description: "Applies a magic function to an input.",
      schema: z.object({
        input: z.string(),
      }),
    }
  );
  const badTools = [badMagicTool];

  const RECURSION_LIMIT = 2 * 2 + 1;
  const appWithBadTools = createReactAgent({ llm, tools: badTools });
  const query = "what is the value of magic_function(3)?";

  try {
    await appWithBadTools.invoke(
      {
        messages: [{ role: "user", content: query }],
      },
      {
        recursionLimit: RECURSION_LIMIT,
      }
    );
  } catch (e) {
    if (e instanceof GraphRecursionError) {
      console.log("Recursion limit reached.");
    } else {
      throw e;
    }
  }
}
```

## Explanation of the Code:

### 1. **Import Statements**:
   - Imports necessary modules from `@langchain/core`, `@langchain/anthropic`, `@langchain/openai`, `@langchain/langgraph`, and `zod`. These modules provide functionalities for creating prompts, parsing outputs, running chains, using chat models, embeddings, similarity calculations, memory management, agent creation, and schema validation.

### 2. **`createSemanticRouterChain` Function**:
   - **Purpose**: Creates a chain that routes queries based on semantic similarity to predefined templates.
   - **Parameters**:
     - `modelName` (string): The name of the Anthropic chat model to use.
     - `templateTexts` (string[]): An array of template texts for different categories (e.g., physics, math).
   - **Implementation**:
     - Initializes `OpenAIEmbeddings` to create embeddings for the templates and queries.
     - Defines an asynchronous function `promptRouter` that:
       - Embeds the input query.
       - Calculates the cosine similarity between the query embedding and the template embeddings.
       - Determines the most relevant template based on the highest similarity score.
       - Returns a `ChatPromptTemplate` using the selected template text.
     - Creates a `ChatAnthropic` model with the specified `modelName`.
     - Constructs a `RunnableSequence` that consists of the `promptRouter`, the `ChatAnthropic` model, and a `StringOutputParser`.
   - **Return Value**: A `RunnableSequence` that performs semantic routing and generates responses.

### 3. **`createLCELRouterChain` Function**:
   - **Purpose**: Creates a chain that routes queries based on a classification step using LangChain Expression Language (LCEL).
   - **Parameters**:
     - `modelName` (string): The name of the Anthropic chat model to use.
   - **Implementation**:
     - Defines a `ChatPromptTemplate` for classifying the input question into categories like "LangChain", "Anthropic", or "Other".
     - Creates a `ChatAnthropic` model with the specified `modelName`.
     - Constructs a `RunnableSequence` (`classificationChain`) that classifies the input question.
     - Defines separate `ChatPromptTemplate` instances (`langChainChain`, `anthropicChain`, `generalChain`) for each category, each piped to the `ChatAnthropic` model.
     - Defines a `route` function that selects the appropriate chain based on the classification result.
     - Creates a `RunnableSequence` (`fullChain`) that first classifies the question and then routes it to the appropriate chain.
   - **Return Value**: A `RunnableSequence` that performs classification-based routing and generates responses.

### 4. **`createLangGraphAgent` Function**:
   - **Purpose**: Creates an agent using LangGraph, demonstrating tool calling and memory.
   - **Parameters**:
     - `modelName` (string): The name of the OpenAI chat model to use.
   - **Implementation**:
     - Defines an OpenAI chat model (`ChatOpenAI`).
     - Defines a tool (`magicTool`) that adds 2 to a given number.
     - Creates an agent (`app`) using `createReactAgent` with the defined LLM and tools.
     - Invokes the agent with a sample query.
     - Demonstrates memory by creating a `MemorySaver` and an agent with memory (`appWithMemory`).
     - Invokes the agent with memory multiple times to show that it retains information from previous interactions.
   - **Return Value**: An object containing the agent's output messages.

### 5. **`createToolCallingAgentExample` Function**:
   - **Purpose**: Creates a tool-calling agent using LangChain agents.
   - **Parameters**:
     - `modelName` (string): The name of the OpenAI chat model to use.
   - **Implementation**:
     - Defines an OpenAI chat model (`ChatOpenAI`).
     - Defines a tool (`magicTool`) that adds 2 to a given number.
     - Defines a prompt using `ChatPromptTemplate`.
     - Creates an agent using `createToolCallingAgent` with the defined LLM, tools, and prompt.
     - Creates an `AgentExecutor` to execute the agent.
     - Invokes the agent with a sample query.
   - **Return Value**: An object containing the agent's output.

### 6. **`createAgentWithMemory` Function**:
   - **Purpose**: Creates an agent with memory using LangChain agents.
   - **Parameters**:
     - `modelName` (string): The name of the OpenAI chat model to use.
   - **Implementation**:
     - Defines an OpenAI chat model (`ChatOpenAI`).
     - Defines a tool (`magicTool`) that adds 2 to a given number.
     - Defines a prompt using `ChatPromptTemplate`.
     - Creates an agent using `createToolCallingAgent` with the defined LLM, tools, and prompt.
     - Creates an `AgentExecutor` to execute the agent.
     - Adds memory to the agent using `ChatMessageHistory` and `RunnableWithMessageHistory`.
     - Invokes the agent multiple times to show that it retains information from previous interactions.
   - **Return Value**: None (void).

### 7. **`createAgentWithMaxIterations` Function**:
   - **Purpose**: Creates an agent with a maximum number of iterations to prevent infinite loops.
   - **Parameters**:
     - `modelName` (string): The name of the OpenAI chat model to use.
   - **Implementation**:
     - Defines an OpenAI chat model (`ChatOpenAI`).
     - Defines a "bad" tool (`badMagicTool`) that always returns an error message.
     - Defines a prompt using `ChatPromptTemplate`.
     - Creates an agent using `createToolCallingAgent` with the defined LLM, tools, and prompt.
     - Creates an `AgentExecutor` with a `maxIterations` parameter set to 2.
     - Invokes the agent with a sample query.
   - **Return Value**: None (void).

### 8. **`createAgentWithRecursionLimit` Function**:
   - **Purpose**: Creates an agent with a recursion limit to prevent infinite loops in LangGraph.
   - **Parameters**:
     - `modelName` (string): The name of the OpenAI chat model to use.
   - **Implementation**:
     - Defines an OpenAI chat model (`ChatOpenAI`).
     - Defines a "bad" tool (`badMagicTool`) that always returns an error message.
     - Creates an agent using `createReactAgent` with the defined LLM and tools.
     - Sets a `recursionLimit` when invoking the agent.
     - Catches `GraphRecursionError` to handle the case where the recursion limit is reached.
   - **Return Value**: None (void).

### Summary:

This module provides several functions for creating RAG chain applications using LangChain.js. It includes examples of:

- **Semantic Routing**: Routing queries based on semantic similarity to predefined templates.
- **LCEL Routing**: Routing queries based on a classification step using LangChain Expression Language.
- **LangGraph Agent**: Creating an agent using LangGraph, demonstrating tool calling and memory.
- **Tool-Calling Agent**: Creating a tool-calling agent using LangChain agents.
- **Agent with Memory**: Creating an agent with memory using LangChain agents.
- **Max Iterations**: Creating an agent with a maximum number of iterations to prevent infinite loops.
- **Recursion Limit**: Creating an agent with a recursion limit to prevent infinite loops in LangGraph.

These examples demonstrate how to use different features of LangChain.js to build sophisticated RAG chain applications.

https://js.langchain.com/docs/how_to/generative_ui
https://js.langchain.com/docs/how_to/migrate_agent
https://js.langchain.com/docs/how_to/routing


In [138]:
from IPython.display import Markdown, display

result = main_chain.invoke("How to call tools with chat models?")
display(Markdown(result['final_output'].content))

for i in result['docs']:
    print(i.metadata['source'])

How to call tools with chat models? datasource='python_docs'
user is asking about python docs


```markdown
To call tools with chat models in LangChain, you need to follow these steps:

1.  **Define Tool Schemas**: You can define tool schemas using:

    *   **Python functions**: Use type hints and docstrings to describe the tool and its arguments.

    ```python
    def add(a: int, b: int) -> int:
        """Add two integers.
        Args:
            a: First integer
            b: Second integer
        """
        return a + b

    def multiply(a: int, b: int) -> int:
        """Multiply two integers.
        Args:
            a: First integer
            b: Second integer
        """
        return a * b
    ```

    *   **Pydantic models**: Define schemas using Pydantic classes.

    ```python
    from pydantic import BaseModel, Field

    class add(BaseModel):
        """Add two integers."""
        a: int = Field(..., description="First integer")
        b: int = Field(..., description="Second integer")

    class multiply(BaseModel):
        """Multiply two integers."""
        a: int = Field(..., description="First integer")
        b: int = Field(..., description="Second integer")
    ```

    *   **TypedDict classes**: Use TypedDicts with annotations.

    ```python
    from typing_extensions import Annotated, TypedDict

    class add(TypedDict):
        """Add two integers."""
        a: Annotated[int, ..., "First integer"]
        b: Annotated[int, ..., "Second integer"]

    class multiply(TypedDict):
        """Multiply two integers."""
        a: Annotated[int, ..., "First integer"]
        b: Annotated[int, ..., "Second integer"]

    tools = [add, multiply]
    ```

2.  **Bind Tools to the Chat Model**: Use the `.bind_tools()` method to associate the tool schemas with the chat model.

    ```python
    pip install -qU "langchain[google-genai]"
    import getpass
    import os

    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

    from langchain.chat_models import init_chat_model

    llm = init_chat_model("gemini-2.0-flash", model_provider="google_genai")

    llm_with_tools = llm.bind_tools(tools)
    ```

3.  **Invoke the Chat Model**: Call the `.invoke()` method with a query. The model will generate arguments for the appropriate tool based on the query.

    ```python
    query = "What is 3 * 12?"
    llm_with_tools.invoke(query)
    ```

    The output will include `tool_calls` with the name of the tool and the arguments.

4.  **Access Tool Calls**: The tool calls are attached to the message as a list of `tool_call` objects in the `.tool_calls` attribute.

    ```python
    query = "What is 3 * 12? Also, what is 11 + 49?"
    llm_with_tools.invoke(query).tool_calls
    ```

5.  **Parsing Tool Calls (Optional)**: Use an output parser like `PydanticToolsParser` to convert the tool calls to Pydantic objects.

    ```python
    from langchain_core.output_parsers import PydanticToolsParser
    from pydantic import BaseModel, Field

    class add(BaseModel):
        """Add two integers."""
        a: int = Field(..., description="First integer")
        b: int = Field(..., description="Second integer")

    class multiply(BaseModel):
        """Multiply two integers."""
        a: int = Field(..., description="First integer")
        b: int = Field(..., description="Second integer")

    chain = llm_with_tools | PydanticToolsParser(tools=[add, multiply])
    chain.invoke(query)
    ```

This process allows the chat model to intelligently decide which tool to use and generate the necessary arguments for it.

https://python.langchain.com/docs/how_to/tool_calling/
https://python.langchain.com/docs/how_to/tool_calling/
https://python.langchain.com/docs/how_to/tool_calling/
